# Bellwether — analytics on Polymarket + robust P&L (Prompt 6)

The same `core` functions from Prompt 4 run unchanged on Polymarket via the
canonical tables (the de-risking payoff). Plus a **cash-flow realized P&L** that
incorporates SPLIT/MERGE/REDEEM/REWARD `position_event` rows.

**Caveats (documented):** the Data API caps deep pagination, so a high-frequency
wallet's history is partial; cash-flow P&L understates by any still-open position
value. Use full (uncapped) loads for production figures.

Prereqs: `tt poly seed-leaderboard`, `tt poly load <wallet>` for a few wallets.

In [ ]:
from bellwether_analytics.core import (
    cashflow_pnl_by_wallet,
    load_events_df,
    load_trades_df,
    performance_by_wallet,
    specialization_by_wallet,
)

trades = load_trades_df(platform="polymarket")
events = load_events_df(platform="polymarket")
print(f"{len(trades)} trades, {len(events)} position events, {trades['wallet'].nunique()} wallets")

In [ ]:
# Same primitives as the Manifold notebook — no code change.
perf = performance_by_wallet(trades)
spec = specialization_by_wallet(trades)
perf.join(spec[["hhi", "top_category"]])

In [ ]:
# Robust cash-flow realized P&L (trades + position events).
pnl = cashflow_pnl_by_wallet(trades, events)
pnl.round(2)

In [ ]:
# Side-by-side comparison across wallets.
compare = perf[["trade_count", "resolved_trade_count"]].join(
    pnl[["buy_cost", "sell_proceeds", "redeemed", "net_cashflow"]]
).join(spec[["hhi", "top_category"]])
compare.sort_values("net_cashflow", ascending=False)